# Lecture 04 — Flow

*Making decisions and repeating work: screening molecules with Lipinski's rule of five.*

## Learning objectives

By the end of this lecture you will be able to:

- write **boolean** expressions with comparisons and `and` / `or` / `not`;
- branch with **`if` / `elif` / `else`**;
- repeat work with **`for`** loops (over lists and over dictionary items);
- use a **`while`** loop, and avoid the infinite-loop trap;
- skip or stop early with **`continue`** / **`break`**;
- write a simple **list comprehension** once the explicit loop is clear.

## Recap of Lecture 03

- A **function** packages a calculation (`def` ... `return`); we built `molar_mass()`.
- Functions take **parameters** (with optional **defaults**) and have **docstrings**.
- `import` brings in libraries; RDKit's `Descriptors.MolWt` matched our hand-built molar mass.

## Booleans and comparisons

A **boolean** is a value that is either `True` or `False`. You get one whenever you **compare** two things:

| Operator | Meaning |
| --- | --- |
| `==` | equal to (note: **two** equals signs) |
| `!=` | not equal to |
| `<`  `>` | less than / greater than |
| `<=`  `>=` | less than or equal / greater than or equal |

In [ ]:
molar_mass = 180.16     # aspirin, g/mol
print(molar_mass < 500)      # is it below 500?
print(molar_mass == 180.16)
print(molar_mass != 500)

Combine conditions with **`and`** (both must be true), **`or`** (at least one), and **`not`** (flip it):

In [ ]:
logp = 1.31     # aspirin's (computed) logP
print(molar_mass < 500 and logp < 5)     # both conditions true?
print(molar_mass > 500 or logp < 5)      # at least one true?
print(not (molar_mass > 500))            # flip a condition

## Making decisions: `if` / `elif` / `else`

An **`if`** statement runs a block of code only when its condition is `True`. Add **`elif`** ("else if") for more cases, and **`else`** for "none of the above". Mind the colons and the indentation — the indented block is what runs.

In [ ]:
mass = 180.16

if mass < 100:
    print("small molecule")
elif mass < 500:
    print("medium-sized molecule")
else:
    print("large molecule")

> **🧪 Chemistry aside — logP and the rule of five**
>
> **logP** is a number describing how *fat-loving* versus *water-loving* a molecule is: higher logP means more fat-loving (it prefers oil to water). **Lipinski's rule of five** is a rule of thumb for whether a molecule is likely to make a good oral drug. It flags a molecule as potentially problematic if it breaks more than one of: molar mass ≤ 500, logP ≤ 5, hydrogen-bond **donors** ≤ 5, hydrogen-bond **acceptors** ≤ 10. We will use it as our worked example for decisions and loops.

## The rule of five, by hand

Let us implement the rule for one molecule using numbers we have looked up. Aspirin: molar mass 180.16, logP 1.31, 1 H-bond donor, 3 H-bond acceptors.

In [ ]:
mass = 180.16
logp = 1.31
donors = 1
acceptors = 3

# Count how many of the four limits are broken.
violations = 0
if mass > 500:
    violations += 1
if logp > 5:
    violations += 1
if donors > 5:
    violations += 1
if acceptors > 10:
    violations += 1

if violations <= 1:
    print(f"Aspirin looks drug-like (violations: {violations})")
else:
    print(f"Aspirin breaks the rule of five (violations: {violations})")

## `for` loops: doing something to every item

A **`for`** loop repeats a block once for each item in a collection. Here we loop over a list of molecule names and print each.

In [ ]:
molecules = ["water", "ethanol", "caffeine", "aspirin", "glucose"]
for name in molecules:
    print("Processing", name)

The loop variable (`name` here) takes each value in turn. We can do real work inside — for example, reuse our `molar_mass()` function from Lecture 03 across a list of molecules described as `{element: count}` dictionaries.

In [ ]:
atomic_mass = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999}

def molar_mass(composition):
    total = 0.0
    for element, count in composition.items():
        total += count * atomic_mass[element]
    return total

# A small list of (name, composition) pairs.
data = [
    ("water",   {"H": 2, "O": 1}),
    ("ethanol", {"C": 2, "H": 6, "O": 1}),
    ("methane", {"C": 1, "H": 4}),
    ("benzene", {"C": 6, "H": 6}),
]

for name, composition in data:
    print(f"{name:<8} {molar_mass(composition):6.2f} g/mol")

### Looping over a dictionary

Looping over a dictionary's `.items()` gives you each **key and value** together — exactly how `molar_mass()` works inside.

In [ ]:
for element, mass in atomic_mass.items():
    print(f"{element}: {mass} g/mol")

### 🔬 Try it yourself

Loop over the `data` list above and find the **heaviest** molecule. (Hint: keep a variable for the biggest mass seen so far, start it at 0, and update it inside the loop when you find something heavier.)

In [ ]:
# Your code here.

**Solution**

In [ ]:
heaviest_name = ""
heaviest_mass = 0.0
for name, composition in data:
    m = molar_mass(composition)
    if m > heaviest_mass:
        heaviest_mass = m
        heaviest_name = name
print(f"The heaviest is {heaviest_name} at {heaviest_mass:.2f} g/mol")

## Filtering with a loop and an `if`

A very common pattern: loop over items and **keep** only those that pass a test. Here we collect molecules below a molar-mass cut-off.

In [ ]:
cutoff = 50.0     # g/mol
light_molecules = []
for name, composition in data:
    if molar_mass(composition) < cutoff:
        light_molecules.append(name)
print(f"Below {cutoff} g/mol:", light_molecules)

## `while` loops (and the infinite-loop trap)

A **`while`** loop keeps going *as long as* a condition stays `True`. Use it when you do not know in advance how many repeats you need. Here we simulate a simple radioactive-style halving until very little remains.

In [ ]:
amount = 100.0      # arbitrary units
half_lives = 0
while amount > 1.0:
    amount = amount / 2     # halve it
    half_lives += 1
print(f"After {half_lives} halvings, {amount:.3f} units remain")

> **⚠️ The infinite-loop trap.** A `while` loop only stops when its condition becomes `False`. If you forget to change anything inside the loop, it runs forever and the notebook hangs. The fix if it happens: **interrupt the kernel** (the stop ◼ button, or *Kernel → Interrupt*). Always make sure something inside the loop moves you towards the stopping condition.

## `break` and `continue`

Inside any loop: **`break`** stops the loop completely; **`continue`** skips to the next item.

In [ ]:
for name, composition in data:
    m = molar_mass(composition)
    if m < 20:
        continue           # skip the very light ones
    print(name, round(m, 2))
    if name == "ethanol":
        print("  (found ethanol — stopping early)")
        break              # stop the whole loop here

## ⚗️ With RDKit — the rule of five on real descriptors

We screened aspirin by hand using looked-up numbers. Now let RDKit compute the four descriptors **live** from SMILES, and run the very same `if` logic on its values. The decision-making is identical — only the source of the numbers changes.

> **🧪 Chemistry aside — an honesty note about logP.** The logP that RDKit gives you (`Descriptors.MolLogP`) is a **computed** value — specifically Crippen's estimate, worked out from the molecule's structure. It is close to, but **not identical to**, the logP you would *measure* experimentally in the lab. That is fine for screening and teaching, but worth remembering: when you see RDKit's logP (here and in the data file), read it as "a good computed estimate", not "the measured truth".

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski

def rule_of_five(smiles):
    "Return (n_violations, is_drug_like) for a molecule given by SMILES."
    mol = Chem.MolFromSmiles(smiles)
    mass = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    donors = Lipinski.NumHDonors(mol)
    acceptors = Lipinski.NumHAcceptors(mol)

    violations = 0
    if mass > 500:
        violations += 1
    if logp > 5:
        violations += 1
    if donors > 5:
        violations += 1
    if acceptors > 10:
        violations += 1
    return violations, violations <= 1

v, drug_like = rule_of_five("CC(=O)Oc1ccccc1C(=O)O")   # aspirin
print(f"Aspirin: {v} violation(s); drug-like? {drug_like}")

Now loop over several molecules (as SMILES) and print each one's verdict — decisions and loops working together:

In [ ]:
smiles_set = {
    "water":      "O",
    "ethanol":    "CCO",
    "caffeine":   "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "aspirin":    "CC(=O)Oc1ccccc1C(=O)O",
    "glucose":    "OCC1OC(O)C(O)C(O)C1O",
    "ibuprofen":  "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
}

for name, smiles in smiles_set.items():
    violations, drug_like = rule_of_five(smiles)
    verdict = "PASS" if drug_like else "FAIL"
    print(f"{name:<10} {violations} violation(s)  ->  {verdict}")

## A gentle first look at list comprehensions

Once you are comfortable writing a `for` loop that builds a list, Python offers a shorthand for that exact pattern: a **list comprehension**. Compare the two — they do the same thing.

In [ ]:
# The explicit loop you already understand:
names_upper = []
for name in smiles_set:
    names_upper.append(name.upper())
print(names_upper)

# The same thing as a list comprehension:
names_upper2 = [name.upper() for name in smiles_set]
print(names_upper2)

Read it as "**give me `name.upper()` for each `name` in `smiles_set`**". You can add a condition too: `[n for n in names if ...]`. Use comprehensions when they make code *clearer*; reach for an ordinary loop whenever you are unsure.

### 🔬 Try it yourself

1. Write a function `is_drug_like(smiles)` that returns just `True` or `False` (reuse `rule_of_five`).
2. Use a **list comprehension** to build a list of the names in `smiles_set` that are drug-like.

In [ ]:
# Your code here.

**Solution**

In [ ]:
def is_drug_like(smiles):
    _, drug_like = rule_of_five(smiles)
    return drug_like

drug_like_names = [name for name, smi in smiles_set.items() if is_drug_like(smi)]
print("Drug-like:", drug_like_names)

## Key takeaways

- **Booleans** come from comparisons (`==`, `<`, `>=`, ...) and combine with `and` / `or` / `not`.
- **`if` / `elif` / `else`** chooses what to do; indentation marks each block.
- **`for`** loops repeat over the items of a list or the `.items()` of a dictionary.
- **`while`** loops repeat until a condition fails — always move towards stopping, or you loop forever.
- **`break`** stops a loop; **`continue`** skips an item; a **list comprehension** is shorthand for a list-building loop.

## Looking ahead

Next lecture — **Files** — we read a real molecule dataset from disk, run this rule-of-five screen across it, and save the drug-like subset to a new file.